<a href="https://colab.research.google.com/github/subod4/ADdetection/blob/main/alzhe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alzheimer's Disease Diagnosis - Complete Improved Pipeline

Based on your original code but with ALL critical improvements:
1. Segmentation usage (extracts only PAR segments)
2. Data augmentation (multiplies dataset)
3. Proper train/val/test split
4. Standardized segment lengths
5. K-fold cross-validation option
6. Test predictions for submission

In [1]:
!pip install gdown
import gdown

!gdown --fuzzy "https://drive.google.com/file/d/1HotQ4CPoHHD-q9y_IVsW8kEEHS78365G/view?usp=sharing" -O /content/train.rar
!gdown --fuzzy "https://drive.google.com/file/d/1jMAH8EpL2vldBRK7jpE1Ln8Ly6occ1Qk/view?usp=sharing" -O /content/test-dist.rar
!unrar x /content/train.rar
!unrar x /content/test-dist

Downloading...
From (original): https://drive.google.com/uc?id=1HotQ4CPoHHD-q9y_IVsW8kEEHS78365G
From (redirected): https://drive.google.com/uc?id=1HotQ4CPoHHD-q9y_IVsW8kEEHS78365G&confirm=t&uuid=edd4e667-b349-4a89-a855-2485c0669762
To: /content/train.rar
100% 997M/997M [00:07<00:00, 132MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1jMAH8EpL2vldBRK7jpE1Ln8Ly6occ1Qk
From (redirected): https://drive.google.com/uc?id=1jMAH8EpL2vldBRK7jpE1Ln8Ly6occ1Qk&confirm=t&uuid=69fd8cd5-f4ee-4995-8e02-053b1704f741
To: /content/test-dist.rar
100% 235M/235M [00:05<00:00, 40.7MB/s]

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/train.rar

Creating    train                                                     OK
Extracting  train/adresso-train-mmse-scores.csv                            0%  OK 
Creating    train/audio                                               OK
Creating    train/audio/ad                               

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from collections import Counter
import pickle
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("="*80)
print("ALZHEIMER'S DISEASE DIAGNOSIS - COMPLETE IMPROVED PIPELINE")
print("="*80)

ALZHEIMER'S DISEASE DIAGNOSIS - COMPLETE IMPROVED PIPELINE


## Configuration

In [ ]:
CONFIG = {
    'sr': 16000,                    # Sample rate
    'segment_duration': 3.0,        # Standardize segments to 3 seconds
    'min_segment_duration': 1.0,    # Minimum segment length to keep
    'n_mels': 128,                  # Number of mel bands
    'fmax': 8000,                   # Maximum frequency
    'use_augmentation': True,       # Enable data augmentation
    'use_kfold': True,              # Use K-fold cross-validation
    'n_folds': 5,                   # Number of folds for cross-validation
    'epochs': 50,                   # Maximum epochs
    'batch_size': 32,               # Batch size
    'learning_rate': 0.0005,        # Learning rate
    'test_folder': '/content/test-dist/audio',
    'test_seg_folder': '/content/test-dist/segmentation'  # Test segmentation folder
}

## 1. Data Loading with Segmentation (CRITICAL IMPROVEMENT!)

This is THE KEY improvement - we extract only patient speech (PAR segments)!

In [ ]:
def augment_audio(y, sr=16000):
    """
    Apply data augmentation to increase dataset diversity.
    Returns list of augmented versions.
    """
    augmented = [y]  # Original

    # Time stretching (90-110% speed)
    if np.random.rand() > 0.5:
        rate = np.random.uniform(0.90, 1.10)
        y_stretch = librosa.effects.time_stretch(y, rate=rate)
        if len(y_stretch) < len(y):
            y_stretch = np.pad(y_stretch, (0, len(y) - len(y_stretch)))
        else:
            y_stretch = y_stretch[:len(y)]
        augmented.append(y_stretch)

    # Pitch shifting (-2 to +2 semitones)
    if np.random.rand() > 0.5:
        n_steps = np.random.randint(-2, 3)
        if n_steps != 0:
            y_pitch = librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
            augmented.append(y_pitch)

    # Add subtle noise
    if np.random.rand() > 0.5:
        noise = np.random.normal(0, 0.003, y.shape)
        y_noise = y + noise
        y_noise = librosa.util.normalize(y_noise)
        augmented.append(y_noise)

    return augmented

def get_file_list(audio_folder, seg_folder):
    """
    Get list of (audio_path, seg_path, filename) tuples for all valid files.
    This is used for file-level splitting BEFORE segment extraction.
    """
    file_list = []
    audio_files = sorted([f for f in os.listdir(audio_folder) if f.endswith('.wav')])

    for filename in audio_files:
        audio_path = os.path.join(audio_folder, filename)
        seg_path = os.path.join(seg_folder, filename.replace('.wav', '.csv'))

        if os.path.exists(seg_path):
            file_list.append((audio_path, seg_path, filename))
        else:
            print(f"  Warning: No segmentation for {filename}")

    return file_list

def extract_segments_from_file(audio_path, seg_path, label, sr=16000,
                                speaker='PAR', min_duration=1.0):
    """
    Extract PAR segments from a SINGLE audio file.
    NO augmentation here - augmentation is applied separately after split.
    """
    segments = []

    try:
        seg_df = pd.read_csv(seg_path)
        y_full, _ = librosa.load(audio_path, sr=sr)

        par_segments = seg_df[seg_df['speaker'] == speaker]

        for _, row in par_segments.iterrows():
            begin_sample = int(row['begin'] * sr / 1000)
            end_sample = int(row['end'] * sr / 1000)

            segment = y_full[begin_sample:end_sample]

            if len(segment) > sr * min_duration:
                segment = librosa.util.normalize(segment)
                segments.append((segment, label))

    except Exception as e:
        print(f"  Error processing {os.path.basename(audio_path)}: {e}")

    return segments

def extract_segments_from_files(file_list, label, sr=16000, min_duration=1.0, augment=False):
    """
    Extract segments from a list of files.
    Augmentation is applied ONLY if augment=True (should be True only for training).
    """
    all_segments = []

    for audio_path, seg_path, filename in file_list:
        segments = extract_segments_from_file(
            audio_path, seg_path, label, sr, min_duration=min_duration
        )

        # Apply augmentation ONLY for training data
        if augment:
            augmented_segments = []
            for segment, lbl in segments:
                aug_versions = augment_audio(segment, sr)
                for aug_seg in aug_versions:
                    augmented_segments.append((aug_seg, lbl))
            all_segments.extend(augmented_segments)
        else:
            all_segments.extend(segments)

    return all_segments

# Legacy function kept for compatibility (but NOT USED in new pipeline)
def load_segmented_audio(audio_file, seg_file, label, sr=16000,
                        speaker='PAR', min_duration=1.0, augment=False):
    """
    [LEGACY - not used in corrected pipeline]
    Load only participant (PAR) segments from audio file.
    """
    return extract_segments_from_file(audio_file, seg_file, label, sr, speaker, min_duration)

def load_all_segmented_audio(audio_folder, seg_folder, label, augment=False):
    """[LEGACY - not used in corrected pipeline]"""
    file_list = get_file_list(audio_folder, seg_folder)
    return extract_segments_from_files(file_list, label, CONFIG['sr'],
                                       CONFIG['min_segment_duration'], augment)

In [ ]:
# ============================================================================
# CORRECTED PIPELINE: Split by FILE first, then extract segments
# This prevents data leakage from same speaker appearing in train AND test
# ============================================================================

print("\n1️⃣  COLLECTING FILE LIST (NO SEGMENTS YET)")
print("-" * 80)

ad_folder = '/content/train/audio/ad'
cn_folder = '/content/train/audio/cn'
ad_seg_folder = '/content/train/segmentation/ad'
cn_seg_folder = '/content/train/segmentation/cn'

print("Collecting AD (Alzheimer's) files...")
ad_files = get_file_list(ad_folder, ad_seg_folder)
print(f"  ✓ Found {len(ad_files)} AD files")

print("\nCollecting CN (Control) files...")
cn_files = get_file_list(cn_folder, cn_seg_folder)
print(f"  ✓ Found {len(cn_files)} CN files")

# Create file-level dataset (before any segmentation!)
all_files = [(f, 'AD') for f in ad_files] + [(f, 'CN') for f in cn_files]
file_labels = ['AD'] * len(ad_files) + ['CN'] * len(cn_files)

print(f"\n📊 TOTAL FILES: {len(all_files)}")
print(f"   AD: {len(ad_files)} files | CN: {len(cn_files)} files")
print(f"\n⚠️  IMPORTANT: Splitting by FILE first to prevent data leakage!")

## 2. Split Files FIRST (Prevent Data Leakage!)

In [ ]:
print("\n2️⃣  SPLITTING FILES FIRST (CRITICAL: Prevents Data Leakage!)")
print("-" * 80)

# Split at FILE level, NOT segment level
# This ensures no speaker/recording appears in both train and test

from sklearn.model_selection import train_test_split

# Separate AD and CN files for stratified splitting
ad_files_list = [(f, 'AD') for f in ad_files]
cn_files_list = [(f, 'CN') for f in cn_files]

# Split AD files: 80% train, 20% test
ad_train, ad_test = train_test_split(
    ad_files_list, test_size=0.20, random_state=42
)

# Split CN files: 80% train, 20% test
cn_train, cn_test = train_test_split(
    cn_files_list, test_size=0.20, random_state=42
)

# Combine
train_files = ad_train + cn_train
test_files = ad_test + cn_test

print(f"✓ FILE-LEVEL SPLIT COMPLETE (stratified)")
print(f"\n  Training files: {len(train_files)} ({len(ad_train)} AD + {len(cn_train)} CN)")
print(f"  Test files:     {len(test_files)} ({len(ad_test)} AD + {len(cn_test)} CN)")

# Show which files are in each split
print(f"\n  Train AD files: {[os.path.basename(f[0][0]) for f in ad_train[:3]]}...")
print(f"  Test AD files:  {[os.path.basename(f[0][0]) for f in ad_test[:3]]}...")
print(f"\n⚠️  No file appears in both train AND test - data leakage prevented!")

## 3. Extract Segments AFTER Split (with Augmentation only for Training)

In [ ]:
print("\n3️⃣  EXTRACTING SEGMENTS AFTER SPLIT")
print("-" * 80)
print("⚠️  Augmentation applied ONLY to training segments!")

# Extract TRAINING segments (WITH augmentation)
print("\nExtracting TRAINING segments (with augmentation)...")
train_segments = []
for (audio_path, seg_path, filename), label in train_files:
    segments = extract_segments_from_file(
        audio_path, seg_path, label,
        CONFIG['sr'], min_duration=CONFIG['min_segment_duration']
    )

    # Apply augmentation ONLY to training data
    if CONFIG['use_augmentation']:
        for segment, lbl in segments:
            aug_versions = augment_audio(segment, CONFIG['sr'])
            for aug_seg in aug_versions:
                train_segments.append((aug_seg, lbl))
    else:
        train_segments.extend(segments)

print(f"  ✓ Training: {len(train_segments)} segments (augmented)")

# Extract TEST segments (NO augmentation - clean evaluation)
print("\nExtracting TEST segments (NO augmentation)...")
test_segments = []
for (audio_path, seg_path, filename), label in test_files:
    segments = extract_segments_from_file(
        audio_path, seg_path, label,
        CONFIG['sr'], min_duration=CONFIG['min_segment_duration']
    )
    test_segments.extend(segments)

print(f"  ✓ Test: {len(test_segments)} segments (no augmentation)")

# Check class distribution
train_labels = [s[1] for s in train_segments]
test_labels = [s[1] for s in test_segments]

print(f"\n📊 SEGMENT DISTRIBUTION:")
print(f"  Training: AD={train_labels.count('AD')}, CN={train_labels.count('CN')}")
print(f"  Test:     AD={test_labels.count('AD')}, CN={test_labels.count('CN')}")

## 4. Standardize Lengths and Extract Spectrograms

In [ ]:
print("\n4️⃣  STANDARDIZING AND EXTRACTING SPECTROGRAMS")
print("-" * 80)

def standardize_segment_length(segment, target_length):
    """Pad or trim to consistent length"""
    if len(segment) < target_length:
        return np.pad(segment, (0, target_length - len(segment)), mode='constant')
    else:
        return segment[:target_length]

def audio_to_melspectrogram(audio, sr=16000, n_mels=128, fmax=8000):
    """Extract Mel spectrogram in dB scale"""
    S = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels, fmax=fmax)
    S_dB = librosa.power_to_db(S, ref=np.max)
    return S_dB

TARGET_LENGTH = int(CONFIG['sr'] * CONFIG['segment_duration'])

# Process TRAINING data
print("Processing training data...")
train_audio = [standardize_segment_length(seg, TARGET_LENGTH) for seg, _ in train_segments]
train_spectrograms = [audio_to_melspectrogram(audio, CONFIG['sr'], CONFIG['n_mels'], CONFIG['fmax'])
                      for audio in train_audio]

# Process TEST data
print("Processing test data...")
test_audio = [standardize_segment_length(seg, TARGET_LENGTH) for seg, _ in test_segments]
test_spectrograms = [audio_to_melspectrogram(audio, CONFIG['sr'], CONFIG['n_mels'], CONFIG['fmax'])
                     for audio in test_audio]

# Encode labels
le = LabelEncoder()
le.fit(['AD', 'CN'])  # Ensure consistent encoding

y_train = le.transform([s[1] for s in train_segments])
y_test = le.transform([s[1] for s in test_segments])

# Prepare arrays with channel dimension
X_train = np.array(train_spectrograms)[..., np.newaxis]
X_test = np.array(test_spectrograms)[..., np.newaxis]

print(f"\n✓ Processing complete!")
print(f"  Spectrogram shape: {train_spectrograms[0].shape}")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_test shape:  {X_test.shape}")
print(f"  y_train: {len(y_train)} labels")
print(f"  y_test:  {len(y_test)} labels")

# Visualize sample spectrograms
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ad_idx = list(y_train).index(le.transform(['AD'])[0])
cn_idx = list(y_train).index(le.transform(['CN'])[0])

librosa.display.specshow(train_spectrograms[ad_idx], sr=CONFIG['sr'],
                        x_axis='time', y_axis='mel', fmax=CONFIG['fmax'], ax=axes[0])
axes[0].set_title('Sample AD Spectrogram (Training)', fontsize=12)
axes[0].set_ylabel('Mel Frequency')

librosa.display.specshow(train_spectrograms[cn_idx], sr=CONFIG['sr'],
                        x_axis='time', y_axis='mel', fmax=CONFIG['fmax'], ax=axes[1])
axes[1].set_title('Sample CN Spectrogram (Training)', fontsize=12)

plt.tight_layout()
plt.show()

## 5. Build Model with Regularization

In [ ]:
print("\n5️⃣  BUILDING CNN MODEL")
print("-" * 80)

def build_cnn_model(input_shape, num_classes=2):
    """
    CNN optimized for small datasets with heavy regularization
    """
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                     input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Dense layers
        layers.Flatten(),
        layers.Dense(128, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

model = build_cnn_model(input_shape=X_train.shape[1:])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\n📊 Trainable parameters: {trainable_params:,}")

## 6. Train Model (with K-Fold Option)

In [ ]:
print("\n6️⃣  TRAINING MODEL")
print("-" * 80)

# NOTE: Since we already split files before segmentation, the train/test split is clean.
# For K-Fold CV, we should use the training segments only.
# The K-fold here operates on training data with a simple validation split.

if CONFIG['use_kfold']:
    print(f"Using {CONFIG['n_folds']}-Fold Cross-Validation on TRAINING data")
    print("=" * 80)

    # For proper CV without leakage, we need to track which file each segment came from
    # Since augmentation creates multiple segments per file, we'll use StratifiedKFold
    # on the training data (which is already file-separated from test)

    kfold = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=42)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train), 1):
        print(f"\nFOLD {fold}/{CONFIG['n_folds']}")
        print("-" * 40)

        # Build fresh model for each fold
        fold_model = build_cnn_model(input_shape=X_train.shape[1:])
        fold_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        # Callbacks
        early_stop = EarlyStopping(monitor='val_loss', patience=10,
                                  restore_best_weights=True, verbose=0)

        # Train
        fold_history = fold_model.fit(
            X_train[train_idx], y_train[train_idx],
            validation_data=(X_train[val_idx], y_train[val_idx]),
            epochs=CONFIG['epochs'],
            batch_size=CONFIG['batch_size'],
            callbacks=[early_stop],
            verbose=0
        )

        # Evaluate
        val_loss, val_acc = fold_model.evaluate(X_train[val_idx], y_train[val_idx], verbose=0)
        fold_scores.append(val_acc)
        print(f"Validation Accuracy: {val_acc:.4f}")

    print("\n" + "=" * 80)
    print("CROSS-VALIDATION RESULTS:")
    print(f"  Mean Accuracy: {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")
    print(f"  Range: [{np.min(fold_scores):.4f}, {np.max(fold_scores):.4f}]")
    print("=" * 80)

    # Train final model on ALL training data
    print("\nTraining final model on full training set...")

    model = build_cnn_model(input_shape=X_train.shape[1:])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=15,
                              restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                 patience=7, min_lr=1e-7, verbose=1)

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=CONFIG['epochs'],
        batch_size=CONFIG['batch_size'],
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

else:
    # Simple training with validation split
    print("Training with validation split (20% of training data)...")

    early_stop = EarlyStopping(monitor='val_loss', patience=15,
                              restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                 patience=7, min_lr=1e-7, verbose=1)

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=CONFIG['epochs'],
        batch_size=CONFIG['batch_size'],
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

## 7. Evaluate on Held-Out Test Set

In [ ]:
print("\n7️⃣  EVALUATION ON HELD-OUT TEST SET")
print("=" * 80)
print("⚠️  Test set contains segments from FILES not seen during training!")
print("    This is a TRUE evaluation without data leakage.\n")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
test_acc = accuracy_score(y_test, y_pred)

print(f"\n🎯 Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap=plt.cm.Blues, values_format='d', ax=ax)
plt.title(f'Confusion Matrix (Test Accuracy: {test_acc:.2%})\n[No Data Leakage - File-Level Split]',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Per-class performance
print("\nPer-Class Accuracy:")
for i, cls in enumerate(le.classes_):
    mask = y_test == i
    if mask.sum() > 0:
        class_acc = accuracy_score(y_test[mask], y_pred[mask])
        print(f"  {cls}: {class_acc:.4f} ({mask.sum()} samples)")

## 7.1 Advanced Model Evaluation Metrics

In [ ]:
print("\n📊 ADVANCED MODEL EVALUATION METRICS")
print("=" * 80)

from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score, roc_auc_score)
from sklearn.calibration import calibration_curve

# Get prediction probabilities for the test set
y_pred_proba_test = model.predict(X_test, verbose=0)

# For binary classification (AD vs CN)
# Assuming class 0 = AD, class 1 = CN (check with le.classes_)
print(f"Classes: {le.classes_}")
print(f"Class 0: {le.classes_[0]}, Class 1: {le.classes_[1]}")

# Get probabilities for the positive class (AD)
# We'll use AD as the positive class for clinical relevance
ad_class_idx = list(le.classes_).index('AD') if 'AD' in le.classes_ else 0
y_scores = y_pred_proba_test[:, ad_class_idx]  # Probability of AD class

# Create binary labels (1 for AD, 0 for CN)
y_true_binary = (y_test == ad_class_idx).astype(int)
y_pred_binary = (y_pred == ad_class_idx).astype(int)

### ROC Curve and AUC

In [ ]:
# ROC Curve
print("\n📈 ROC CURVE AND AUC")
print("-" * 50)

fpr, tpr, thresholds_roc = roc_curve(y_true_binary, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.3, color='darkorange')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title(f'Receiver Operating Characteristic (ROC) Curve\nAUC = {roc_auc:.4f}', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)

# Add optimal threshold point (Youden's J statistic)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds_roc[optimal_idx]
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], marker='o', color='red', s=100,
            label=f'Optimal Threshold ({optimal_threshold:.3f})', zorder=5)
plt.legend(loc="lower right", fontsize=11)

plt.tight_layout()
plt.show()

print(f"✓ AUC-ROC Score: {roc_auc:.4f}")
print(f"✓ Optimal Threshold (Youden's J): {optimal_threshold:.4f}")

### Precision-Recall (PR) Curve

In [ ]:
# Precision-Recall Curve
print("\n📉 PRECISION-RECALL CURVE")
print("-" * 50)

precision, recall, thresholds_pr = precision_recall_curve(y_true_binary, y_scores)
avg_precision = average_precision_score(y_true_binary, y_scores)

# Calculate baseline (proportion of positive class)
baseline = np.sum(y_true_binary) / len(y_true_binary)

plt.figure(figsize=(10, 8))
plt.plot(recall, precision, color='green', lw=2, label=f'PR curve (AP = {avg_precision:.4f})')
plt.axhline(y=baseline, color='red', linestyle='--', lw=2, label=f'Baseline (No Skill) = {baseline:.3f}')
plt.fill_between(recall, precision, alpha=0.3, color='green')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall (Sensitivity)', fontsize=12)
plt.ylabel('Precision (Positive Predictive Value)', fontsize=12)
plt.title(f'Precision-Recall Curve\nAverage Precision = {avg_precision:.4f}', fontsize=14, fontweight='bold')
plt.legend(loc="upper right", fontsize=11)
plt.grid(True, alpha=0.3)

# Find threshold for best F1 score
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_f1_idx = np.argmax(f1_scores[:-1])  # Exclude last value (recall=0)
plt.scatter(recall[best_f1_idx], precision[best_f1_idx], marker='o', color='red', s=100,
            label=f'Best F1 ({f1_scores[best_f1_idx]:.3f})', zorder=5)
plt.legend(loc="upper right", fontsize=11)

plt.tight_layout()
plt.show()

print(f"✓ Average Precision (AP) Score: {avg_precision:.4f}")
print(f"✓ Best F1 Score: {f1_scores[best_f1_idx]:.4f}")

### Sensitivity and Specificity Analysis

In [ ]:
# Sensitivity and Specificity Analysis
print("\n🔬 SENSITIVITY / SPECIFICITY ANALYSIS")
print("-" * 50)

# Calculate confusion matrix components
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true_binary, y_pred_binary)
TN, FP, FN, TP = cm.ravel()

# Calculate metrics
sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0  # True Positive Rate (Recall)
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0  # True Negative Rate
ppv = TP / (TP + FP) if (TP + FP) > 0 else 0          # Positive Predictive Value (Precision)
npv = TN / (TN + FN) if (TN + FN) > 0 else 0          # Negative Predictive Value
f1 = 2 * (ppv * sensitivity) / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0
balanced_acc = (sensitivity + specificity) / 2

print(f"\n📊 Confusion Matrix Components:")
print(f"   True Positives (TP):  {TP}")
print(f"   True Negatives (TN):  {TN}")
print(f"   False Positives (FP): {FP}")
print(f"   False Negatives (FN): {FN}")

print(f"\n📈 Performance Metrics (AD as positive class):")
print(f"   Sensitivity (Recall/TPR):     {sensitivity:.4f} ({sensitivity*100:.2f}%)")
print(f"   Specificity (TNR):            {specificity:.4f} ({specificity*100:.2f}%)")
print(f"   Precision (PPV):              {ppv:.4f} ({ppv*100:.2f}%)")
print(f"   Negative Predictive Value:    {npv:.4f} ({npv*100:.2f}%)")
print(f"   F1 Score:                     {f1:.4f}")
print(f"   Balanced Accuracy:            {balanced_acc:.4f} ({balanced_acc*100:.2f}%)")

# Sensitivity vs Specificity trade-off plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Sensitivity and Specificity at different thresholds
thresholds_plot = np.linspace(0, 1, 100)
sensitivities = []
specificities = []

for thresh in thresholds_plot:
    y_pred_thresh = (y_scores >= thresh).astype(int)
    cm_thresh = confusion_matrix(y_true_binary, y_pred_thresh, labels=[0, 1])
    if cm_thresh.shape == (2, 2):
        tn, fp, fn, tp = cm_thresh.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        sens = 0
        spec = 0
    sensitivities.append(sens)
    specificities.append(spec)

axes[0].plot(thresholds_plot, sensitivities, 'b-', lw=2, label='Sensitivity (TPR)')
axes[0].plot(thresholds_plot, specificities, 'r-', lw=2, label='Specificity (TNR)')
axes[0].axvline(x=0.5, color='gray', linestyle='--', lw=1, label='Default Threshold (0.5)')
axes[0].axvline(x=optimal_threshold, color='green', linestyle='--', lw=1, label=f'Optimal Threshold ({optimal_threshold:.3f})')
axes[0].set_xlabel('Classification Threshold', fontsize=12)
axes[0].set_ylabel('Rate', fontsize=12)
axes[0].set_title('Sensitivity vs Specificity Trade-off', fontsize=14, fontweight='bold')
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])

# Plot 2: Bar chart of metrics
metrics_names = ['Sensitivity', 'Specificity', 'Precision', 'NPV', 'F1 Score', 'Balanced\nAccuracy']
metrics_values = [sensitivity, specificity, ppv, npv, f1, balanced_acc]
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12', '#1abc9c']

bars = axes[1].bar(metrics_names, metrics_values, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Model Performance Metrics Summary', fontsize=14, fontweight='bold')
axes[1].set_ylim([0, 1.1])
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random Baseline')
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars, metrics_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### Calibration Analysis

In [ ]:
# Calibration Analysis
print("\n📐 CALIBRATION ANALYSIS")
print("-" * 50)

from sklearn.metrics import brier_score_loss

# Calculate calibration curve
n_bins = 10
prob_true, prob_pred = calibration_curve(y_true_binary, y_scores, n_bins=n_bins, strategy='uniform')

# Calculate Brier Score (lower is better)
brier_score = brier_score_loss(y_true_binary, y_scores)

# Expected Calibration Error (ECE)
bin_counts = np.histogram(y_scores, bins=n_bins, range=(0, 1))[0]
ece = np.sum(np.abs(prob_true - prob_pred) * bin_counts[:len(prob_true)]) / np.sum(bin_counts[:len(prob_true)]) if len(prob_true) > 0 else 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Calibration Curve (Reliability Diagram)
axes[0].plot([0, 1], [0, 1], 'k--', lw=2, label='Perfectly Calibrated')
axes[0].plot(prob_pred, prob_true, 'o-', color='darkorange', lw=2, markersize=8,
             label=f'Model (Brier Score = {brier_score:.4f})')
axes[0].fill_between(prob_pred, prob_true, prob_pred, alpha=0.2, color='darkorange')
axes[0].set_xlabel('Mean Predicted Probability', fontsize=12)
axes[0].set_ylabel('Fraction of Positives (True Probability)', fontsize=12)
axes[0].set_title('Calibration Curve (Reliability Diagram)', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# Plot 2: Prediction Distribution with Calibration Info
axes[1].hist(y_scores[y_true_binary == 0], bins=20, alpha=0.6, color='blue',
             label=f'CN (n={np.sum(y_true_binary == 0)})', density=True)
axes[1].hist(y_scores[y_true_binary == 1], bins=20, alpha=0.6, color='red',
             label=f'AD (n={np.sum(y_true_binary == 1)})', density=True)
axes[1].axvline(x=0.5, color='black', linestyle='--', lw=2, label='Decision Threshold (0.5)')
axes[1].axvline(x=optimal_threshold, color='green', linestyle='--', lw=2,
                label=f'Optimal Threshold ({optimal_threshold:.3f})')
axes[1].set_xlabel('Predicted Probability for AD', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Distribution of Predicted Probabilities', fontsize=14, fontweight='bold')
axes[1].legend(loc='upper center', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Calibration Metrics:")
print(f"   Brier Score: {brier_score:.4f} (0 = perfect, 0.25 = random)")
print(f"   Expected Calibration Error (ECE): {ece:.4f}")

# Interpretation
if brier_score < 0.1:
    calibration_quality = "Excellent"
elif brier_score < 0.2:
    calibration_quality = "Good"
elif brier_score < 0.25:
    calibration_quality = "Fair"
else:
    calibration_quality = "Poor"

print(f"\n   Calibration Quality: {calibration_quality}")
print(f"\n💡 Interpretation:")
print(f"   - Brier Score measures the mean squared error between predicted")
print(f"     probabilities and actual outcomes (lower = better)")
print(f"   - A well-calibrated model's curve should follow the diagonal")
print(f"   - Curves above diagonal: model underestimates probability")
print(f"   - Curves below diagonal: model overestimates probability")

### Combined ROC and PR Curves with Summary

In [ ]:
# Combined Summary Plot
print("\n📊 COMPREHENSIVE EVALUATION SUMMARY")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: ROC Curve
axes[0, 0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 0].fill_between(fpr, tpr, alpha=0.3, color='darkorange')
axes[0, 0].scatter(fpr[optimal_idx], tpr[optimal_idx], marker='*', color='red', s=150, zorder=5)
axes[0, 0].set_xlabel('False Positive Rate', fontsize=11)
axes[0, 0].set_ylabel('True Positive Rate', fontsize=11)
axes[0, 0].set_title(f'ROC Curve (AUC = {roc_auc:.4f})', fontsize=12, fontweight='bold')
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: PR Curve
axes[0, 1].plot(recall, precision, color='green', lw=2, label=f'PR (AP = {avg_precision:.4f})')
axes[0, 1].axhline(y=baseline, color='red', linestyle='--', lw=1)
axes[0, 1].fill_between(recall, precision, alpha=0.3, color='green')
axes[0, 1].set_xlabel('Recall', fontsize=11)
axes[0, 1].set_ylabel('Precision', fontsize=11)
axes[0, 1].set_title(f'Precision-Recall Curve (AP = {avg_precision:.4f})', fontsize=12, fontweight='bold')
axes[0, 1].legend(loc='upper right')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Calibration Curve
axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=2)
axes[1, 0].plot(prob_pred, prob_true, 'o-', color='purple', lw=2, markersize=8)
axes[1, 0].set_xlabel('Mean Predicted Probability', fontsize=11)
axes[1, 0].set_ylabel('Fraction of Positives', fontsize=11)
axes[1, 0].set_title(f'Calibration Curve (Brier = {brier_score:.4f})', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Summary Metrics Table
axes[1, 1].axis('off')
summary_data = [
    ['Metric', 'Value'],
    ['AUC-ROC', f'{roc_auc:.4f}'],
    ['Average Precision', f'{avg_precision:.4f}'],
    ['Sensitivity (Recall)', f'{sensitivity:.4f}'],
    ['Specificity', f'{specificity:.4f}'],
    ['Precision (PPV)', f'{ppv:.4f}'],
    ['F1 Score', f'{f1:.4f}'],
    ['Balanced Accuracy', f'{balanced_acc:.4f}'],
    ['Brier Score', f'{brier_score:.4f}'],
    ['Optimal Threshold', f'{optimal_threshold:.4f}']
]

table = axes[1, 1].table(cellText=summary_data[1:], colLabels=summary_data[0],
                          loc='center', cellLoc='center',
                          colColours=['#4CAF50', '#4CAF50'],
                          colWidths=[0.5, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.8)

# Style header
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight='bold', color='white')
        cell.set_facecolor('#2E7D32')
    else:
        if col == 1:
            cell.set_facecolor('#E8F5E9')

axes[1, 1].set_title('Performance Summary', fontsize=14, fontweight='bold', y=0.95)

plt.tight_layout()
plt.savefig('model_evaluation_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Evaluation Summary:")
print(f"   • AUC-ROC: {roc_auc:.4f} - {'Excellent' if roc_auc > 0.9 else 'Good' if roc_auc > 0.8 else 'Fair' if roc_auc > 0.7 else 'Poor'}")
print(f"   • Average Precision: {avg_precision:.4f}")
print(f"   • Sensitivity: {sensitivity:.4f} (ability to detect AD)")
print(f"   • Specificity: {specificity:.4f} (ability to identify CN)")
print(f"   • Calibration: {calibration_quality}")
print(f"\n💾 Summary plot saved to: model_evaluation_summary.png")

## 8. Visualize Training History

In [ ]:
print("\n8️⃣  TRAINING HISTORY")
print("-" * 80)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Load and Predict on Test-Dist Data

## 10. Save Predictions to CSV

## 11. Save Model and Preprocessing Parameters

In [ ]:
print("\n1️⃣1️⃣  SAVING MODEL AND PARAMETERS")
print("-" * 80)

# Save model
model.save('alzheimers_cnn_model_improved.keras')
print("✓ Model saved: alzheimers_cnn_model_improved.keras")

# Save label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print("✓ Label encoder saved: label_encoder.pkl")

# Save preprocessing parameters
preprocessing_params = {
    'sr': CONFIG['sr'],
    'segment_duration': CONFIG['segment_duration'],
    'n_mels': CONFIG['n_mels'],
    'fmax': CONFIG['fmax'],
    'target_length': TARGET_LENGTH,
    'spectrogram_shape': train_spectrograms[0].shape  # Use train_spectrograms
}
with open('preprocessing_params.pkl', 'wb') as f:
    pickle.dump(preprocessing_params, f)
print("✓ Preprocessing params saved: preprocessing_params.pkl")

print("\n" + "=" * 80)
print("✅ PIPELINE COMPLETE!")
print("=" * 80)
print(f"Final Test Accuracy: {test_acc:.2%}")
print(f"Total Training Segments: {len(X_train)}")
print(f"Total Test Segments: {len(X_test)}")
print(f"Test Predictions: {len(test_ids)} files")
print("=" * 80)
print("\n🛡️  DATA INTEGRITY:")
print("   • File-level split: Speakers in test never seen during training")
print("   • Augmentation: Applied ONLY to training data")
print("   • No data leakage between train and test sets")